# Prática 24/11 · Perfis de municípios de MT: agrupamento com K-Means
**Aprendizagem Computacional · 4º TIA · Aula 05 · versão INICIAL**

**Pergunta:** os 142 municípios de MT se dividem em poucos **tipos** de eleitorado? Uma Secretaria que fosse planejar campanhas de participação poderia desenhar uma ação por tipo, em vez de 142 ações diferentes.
Hoje **não há y**: ninguém diz ao modelo qual é o grupo certo. É aprendizagem **não supervisionada** (cap. 19 e 20).

| Parte | O que você faz | Apostila |
|---|---|---|
| 1 | Carregar (pronto) | — |
| 2 | **TODO 1:** escolher as colunas e padronizar | cap. 6, 19 |
| 3 | **TODO 2:** PCA para enxergar em 2D | cap. 7 |
| 4 | **TODO 3:** testar k de 2 a 8 (cotovelo e silhueta) | cap. 19, 20 |
| 5 | **TODO 4:** o agrupamento final e o perfil de cada grupo | cap. 20, 24 |
| 6 | **TODO 5:** o "rótulo guardado": os grupos dizem algo sobre a abstenção? | cap. 19 |
| 7 | Verificação final | — |

Não precisa instalar nada no Colab. O notebook não grava nenhum arquivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
FEATURES = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO']
pd.set_option('display.precision', 2)

## 1 · Carregar os dados (pronto)
Os 142 municípios de MT (TSE, 2026). Vamos agrupar **só pelo perfil do eleitorado**. Abstenção e brancos e nulos ficam guardados para a parte 6.

In [ ]:
ARQUIVO = 'municipios_mt_votos.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-24_Aula05_Perfis_Municipios/dados/municipios_mt_votos.csv'

candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
if local is not None:
    municipios = pd.read_csv(local)
    print('Lido do arquivo:', local.resolve())
else:
    municipios = pd.read_csv(URL)
    print('Lido do GitHub da disciplina')
print(municipios.shape)

## 2 · TODO 1: colunas e padronização (cap. 6 e 19)
O K-Means agrupa por **distância**. Sem padronizar, a coluna de maior escala manda sozinha (cap. 6).
1. `X = municipios[FEATURES]`;
2. `Z = StandardScaler().fit_transform(X)`: cada coluna passa a ter média 0 e desvio 1.

**Não há treino e teste aqui.** Sem y, não existe "acerto" para medir num teste. A pergunta é se os grupos são **coerentes** e **úteis** (cap. 19).

Analogia: é a **estratificação** do inventário florestal. Antes de amostrar, dividimos a área em estratos homogêneos (idade, sítio, espaçamento) para que cada estrato seja tratado de um jeito.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO 1
X = None
Z = None

print('formato:', Z.shape)
print('médias depois de padronizar:', Z.mean(axis=0).round(2))
print('desvios depois de padronizar:', Z.std(axis=0).round(2))

## 3 · TODO 2: PCA para enxergar em 2D (cap. 7)
Com 4 colunas não dá para desenhar. O **PCA** cria 2 eixos novos (componentes) que guardam a maior parte da variação.
Crie `pca = PCA(n_components=2, random_state=SEMENTE)` e `Z2 = pca.fit_transform(Z)`.

In [ ]:
from sklearn.decomposition import PCA

# TODO 2
pca = None
Z2 = None

print('variância explicada por componente:', pca.explained_variance_ratio_.round(3))
print(f'as 2 juntas guardam {pca.explained_variance_ratio_.sum():.0%} da variação')
print('\npeso de cada coluna nos componentes:')
print(pd.DataFrame(pca.components_, columns=FEATURES, index=['PC1', 'PC2']).round(2))

plt.figure(figsize=(6, 4.5))
plt.scatter(Z2[:, 0], Z2[:, 1], s=18, color='grey')
for nome in ['CUIABÁ', 'VÁRZEA GRANDE', 'SINOP', 'COLNIZA', 'CHAPADA DOS GUIMARÃES']:
    i = municipios.index[municipios['NM_MUNICIPIO'] == nome][0]
    plt.annotate(nome.title(), Z2[i], fontsize=8)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Municípios de MT no plano do PCA'); plt.show()

## 4 · TODO 3: quantos grupos? (cap. 19 e 20)
O K-Means precisa que **nós** digamos o k. Vamos testar de 2 a 8 com duas réguas:
- **inércia:** soma das distâncias de cada município ao centro do seu grupo. Sempre cai com mais grupos; procuramos o "cotovelo", onde ela para de cair rápido;
- **silhueta:** de −1 a 1, compara o quanto cada município está perto do próprio grupo e longe do vizinho. **Maior é melhor.**

Dentro do `for`: `km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(Z)`, e depois `km.inertia_` e `silhouette_score(Z, km.labels_)`.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

linhas = []
for k in range(2, 9):
    # TODO 3
    km = None
    linhas.append({'k': k, 'inércia': None, 'silhueta': None})

escolha_k = pd.DataFrame(linhas).set_index('k')
print(escolha_k.round(3))

fig, eixos = plt.subplots(1, 2, figsize=(10, 3.3))
escolha_k['inércia'].plot(ax=eixos[0], marker='o', title='Cotovelo (inércia)')
escolha_k['silhueta'].plot(ax=eixos[1], marker='o', color='C1', title='Silhueta média')
plt.tight_layout(); plt.show()

**Leitura:** a silhueta é máxima em **k = 3**, e a inércia faz o cotovelo mais visível ali também. Uma silhueta de 0,30 indica grupos reais, mas com fronteiras borradas: há municípios "no meio do caminho". É o normal em dados sociais.

## 5 · TODO 4: o agrupamento final e o perfil de cada grupo (cap. 20 e 24)
1. ajuste `km_final = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit(Z)`;
2. guarde os rótulos numa coluna: `municipios['GRUPO'] = km_final.labels_`.

O resto (perfil médio e maiores municípios de cada grupo) está pronto.

In [ ]:
# TODO 4
km_final = None
municipios['GRUPO'] = None

perfil = municipios.groupby('GRUPO')[FEATURES].mean()
perfil['municípios'] = municipios['GRUPO'].value_counts()
print(perfil.round(2))
print()
for g in sorted(municipios['GRUPO'].unique()):
    maiores = municipios[municipios['GRUPO'] == g].nlargest(4, 'QT_APTOS')['NM_MUNICIPIO'].str.title()
    print(f'grupo {g}:', ', '.join(maiores))

plt.figure(figsize=(6, 4.5))
plt.scatter(Z2[:, 0], Z2[:, 1], c=municipios['GRUPO'], cmap='viridis', s=20)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Os 3 grupos no plano do PCA'); plt.show()

**Pergunta para discussão:** que nome você daria a cada grupo, olhando o perfil médio e os maiores municípios?

**Resposta esperada (com estes dados):** um grupo de **polos urbanos** (Cuiabá, Várzea Grande, Rondonópolis, Sinop: eleitorado grande e mais escolarizado), um de **municípios jovens** (Colniza, Aripuanã: mais eleitores de 16 a 24 anos) e um de **municípios envelhecidos** (Jaciara, Chapada dos Guimarães: mais eleitores com 65 anos ou mais). O número do grupo (0, 1, 2) é arbitrário: com outra semente, os grupos podem vir com números trocados, e alguns municípios de fronteira mudam de grupo. O que importa é o perfil.

## 6 · TODO 5: o "rótulo guardado" (cap. 19)
A abstenção e os brancos e nulos **não** entraram no agrupamento. Se os grupos forem úteis, eles devem diferir nessas colunas também.
Calcule `por_grupo = municipios.groupby('GRUPO')[['ABSTENCAO_PCT', 'PCT_BRANCOS_NULOS']].mean()`.

In [ ]:
# TODO 5
por_grupo = None

print(por_grupo.round(2))

**Pergunta para discussão:** os grupos diferem na abstenção e nos brancos e nulos? Isso "valida" o agrupamento?

**Resposta esperada:** diferem. O grupo dos municípios jovens tem a maior abstenção (cerca de 24,5%, contra 22,6% a 22,8% nos outros dois), e o dos envelhecidos tem mais brancos e nulos (cerca de 3,1%, contra 2,7%). O agrupamento captou algo relevante para a decisão sem ter visto essas colunas: a campanha contra a abstenção começaria pelo grupo jovem, e a orientação sobre como votar, pelo envelhecido. Mas isso **não** prova que os grupos estejam "certos": num agrupamento não há resposta certa. Há grupos mais ou menos úteis para um objetivo. No projeto da PPI, justifique o k (silhueta) **e** a utilidade (o que muda na decisão para cada grupo).

## 7 · Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('Z padronizado: 142 × 4, médias 0 e desvios 1', lambda: Z.shape == (142, 4) and np.allclose(Z.mean(axis=0), 0) and np.allclose(Z.std(axis=0), 1)),
    conferir('PCA: as 2 componentes guardam 83% da variação', lambda: round(pca.explained_variance_ratio_.sum(), 2) == 0.83),
    conferir('silhueta máxima em k = 3 (0,30)', lambda: escolha_k['silhueta'].idxmax() == 3 and round(escolha_k.loc[3, 'silhueta'], 2) == 0.30),
    conferir('grupos com 31, 50 e 61 municípios', lambda: sorted(municipios['GRUPO'].value_counts()) == [31, 50, 61]),
    conferir('tabela por grupo tem abstenção e brancos/nulos', lambda: list(por_grupo.columns) == ['ABSTENCAO_PCT', 'PCT_BRANCOS_NULOS'] and len(por_grupo) == 3),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as 6 perguntas da PPI, versão não supervisionada)
1. **Qual é o y?** Não há: o objetivo é descrever o território em tipos. 2. **Baseline?** "Um grupo só" (tratar todos iguais). A pergunta é se os 3 grupos mudam a decisão.
3. **Métrica?** silhueta para escolher o k, e o perfil para dar sentido. 4. **Split?** Não há teste. A checagem é com colunas guardadas (parte 6).
5. **Vazamento?** abstenção e brancos e nulos ficaram fora do agrupamento de propósito. 6. **Evidência?** silhueta de 0,30: grupos moderados. Diga isso.

**Para quem terminar:** troque o K-Means por `AgglomerativeClustering(n_clusters=3)` (de `sklearn.cluster`) e compare os grupos com `pd.crosstab(municipios['GRUPO'], novos_rotulos)`. Os dois métodos concordam? (cap. 20)